## **Customer Lifetime Value (CLV)**

In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")      

In [0]:
%sql
create or replace view identifier(:catalog).gold.customer_lifetime_value as
WITH customer_orders AS (

    SELECT
        c.customer_id,
        c.name AS customer_name,
        s.sale_date,
        s.total_amount,

        FIRST_VALUE(s.sale_date) OVER (
            PARTITION BY c.customer_id
            ORDER BY s.sale_date
        ) AS first_purchase_date

    FROM identifier(:catalog).silver.sales_clean s
    JOIN identifier(:catalog).silver.customers_clean c
        ON s.customer_id = c.customer_id

)

SELECT
    customer_id,
    customer_name,
    first_purchase_date,
    ROUND(SUM(total_amount), 2) AS customer_lifetime_value
FROM customer_orders
GROUP BY
    customer_id,
    customer_name,
    first_purchase_date
ORDER BY customer_lifetime_value DESC;

## **New vs Returning Customers**

In [0]:
%sql
create or replace view identifier(:catalog).gold.newAndReturningCustomers as
WITH customer_orders AS (

    SELECT
        customer_id,
        sale_date,

        LAG(sale_date) OVER (
            PARTITION BY customer_id
            ORDER BY sale_date
        ) AS previous_order

    FROM identifier(:catalog).silver.sales_clean

)

SELECT

    CASE
        WHEN previous_order IS NULL
        THEN 'New Customer'

        ELSE 'Returning Customer'

    END AS customer_type,

    COUNT(*) AS total_orders

FROM customer_orders

GROUP BY customer_type;

## **Average Order Value by Customer Segment**

In [0]:
%sql
create or replace view identifier(:catalog).gold.customerSegments as
WITH customer_revenue AS (

    SELECT

        customer_id,

        round(AVG(total_amount), 2) AS avg_order_value,

        CASE

            WHEN round(AVG(total_amount), 2) >= 30000
            THEN 'Premium'

            ELSE 'Standard'

        END AS customer_segment

    FROM identifier(:catalog).silver.sales_clean

    GROUP BY customer_id

),

segment_orders AS (

    SELECT

        customer_id,

        customer_segment,

        avg_order_value,

        LEAD(avg_order_value) OVER (

            PARTITION BY customer_segment

            ORDER BY avg_order_value DESC

        ) AS next_customer_avg

    FROM customer_revenue

)

SELECT *

FROM segment_orders

ORDER BY
customer_segment,
avg_order_value DESC

In [0]:
%sql
create or replace view identifier(:catalog).gold.Cohort_retention as
WITH customer_cohort AS (

SELECT

customer_id,

DATE_TRUNC('month', signup_date) AS cohort_month

FROM identifier(:catalog).silver.customers_clean

),

customer_activity AS (

SELECT

customer_id,

DATE_TRUNC('month', sale_date) AS purchase_month

FROM identifier(:catalog).silver.sales_clean

),

cohort_analysis AS (

SELECT

c.customer_id,

c.cohort_month,

a.purchase_month,

CAST(
MONTHS_BETWEEN(
a.purchase_month,
c.cohort_month
) AS INT
) AS months_since_signup

FROM customer_cohort c

JOIN customer_activity a

ON c.customer_id=a.customer_id

)

SELECT

cohort_month,

months_since_signup,

COUNT(DISTINCT customer_id) active_customers

FROM cohort_analysis

GROUP BY

cohort_month,

months_since_signup

ORDER BY

cohort_month,

months_since_signup;